# Replikasi Metodologi Artikel (Standalone Colab Notebook)

Semua code, konfigurasi, dan dependency ada di notebook ini.
Tidak perlu `requirements.txt` atau `config/experiment.yaml`.


## 1) Install dependencies (inline)

In [5]:
%pip install -q pandas scikit-learn matplotlib scipy joblib


## 2) Set root folder output

In [6]:
from pathlib import Path
import os

ROOT = Path('/content/task2_replicate')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print('Working dir:', Path.cwd())


Working dir: /content/task2_replicate


## 3) Full pipeline code (inline config + inline logic)

In [7]:
import json
import hashlib
import shutil
import re
from pathlib import Path
from urllib.request import urlopen
from urllib.error import HTTPError, URLError

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import randint, loguniform, uniform

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    cohen_kappa_score,
    matthews_corrcoef,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier


def default_cfg() -> dict:
    return {
        "project": {
            "name": "Repplikasi Metodologi Prediksi Penyakit Jantung",
            "language": "id",
            "random_state": 42,
        },
        "paths": {
            "raw_data": "data/raw/statlog_cleveland_hungary_final.csv",
            "processed_data": "data/processed",
            "tables": "results/tables",
            "figures": "results/figures",
            "submission_dir": "submission_bundle",
            "submission_zip": "submission_bundle.zip",
        },
        "data": {
            "dataset_urls": [
                "https://raw.githubusercontent.com/akmand/datasets/main/heart_statlog_cleveland_hungary_final.csv",
                "https://raw.githubusercontent.com/mrdbourke/zero-to-mastery-ml/master/data/heart-disease.csv",
            ],
            "test_size": 0.2,
        },
        "features": {
            "target_candidates": ["target", "heartdisease", "num", "class", "label", "output"],
            "categorical_candidates": [
                "sex", "cp", "chestpaintype", "fbs", "fastingbs", "restecg",
                "exerciseangina", "exang", "slope", "stslope", "thal", "ca",
            ],
        },
        "cv": {"folds": 5, "shuffle": True},
        "search": {"random_n_iter": 40},
        "models": {"enable_stacking_grid": True},
    }


def ensure_dirs(root: Path, cfg: dict) -> None:
    for key in ["processed_data", "tables", "figures", "submission_dir"]:
        (root / cfg["paths"][key]).mkdir(parents=True, exist_ok=True)


def normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", name.lower())


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def try_download(url: str, path: Path) -> bool:
    try:
        with urlopen(url, timeout=30) as r:
            payload = r.read()
        path.write_bytes(payload)
        return True
    except (HTTPError, URLError, TimeoutError, ValueError):
        return False


def locate_or_download_dataset(root: Path, cfg: dict) -> tuple[Path, str, str]:
    path = root / cfg["paths"]["raw_data"]
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists() and path.stat().st_size > 0:
        return path, "local", "existing file"

    for url in cfg["data"].get("dataset_urls", []):
        if try_download(url, path):
            return path, "download", url

    raise FileNotFoundError(
        "Dataset tidak ditemukan via URL. Upload manual file CSV ke: "
        f"{path}"
    )


def resolve_target_col(df: pd.DataFrame, candidates: list[str]) -> str:
    n2c = {normalize_name(c): c for c in df.columns}
    for c in candidates:
        n = normalize_name(c)
        if n in n2c:
            return n2c[n]
    raise KeyError(f"Target tidak ditemukan. Kolom tersedia: {list(df.columns)}")


def coerce_binary_target(y: pd.Series) -> pd.Series:
    if y.dtype == object:
        mapped = y.astype(str).str.strip().str.lower().map(
            {
                "1": 1, "0": 0,
                "yes": 1, "no": 0,
                "true": 1, "false": 0,
                "positive": 1, "negative": 0,
                "present": 1, "absent": 0,
                "disease": 1, "nodisease": 0,
            }
        )
        if mapped.isna().any():
            uniq = sorted(y.dropna().astype(str).unique())
            if len(uniq) == 2:
                return y.astype(str).map({uniq[0]: 0, uniq[1]: 1}).astype(int)
            raise ValueError("Target kategorikal tidak bisa dipetakan biner")
        return mapped.astype(int)

    uniq = pd.Series(y.dropna().unique())
    if set(uniq.tolist()) <= {0, 1}:
        return y.astype(int)
    return (y.astype(float) > 0).astype(int)


def resolve_feature_groups(X: pd.DataFrame, cat_candidates: list[str]) -> tuple[list[str], list[str]]:
    n2c = {normalize_name(c): c for c in X.columns}

    cat_cols = []
    for c in cat_candidates:
        n = normalize_name(c)
        if n in n2c:
            col = n2c[n]
            if col not in cat_cols:
                cat_cols.append(col)

    if not cat_cols:
        for c in X.columns:
            if pd.api.types.is_integer_dtype(X[c]) and X[c].nunique(dropna=True) <= 10:
                cat_cols.append(c)

    cat_cols = [c for c in cat_cols if c in X.columns]
    num_cols = [c for c in X.columns if c not in cat_cols]
    return num_cols, cat_cols


def build_preprocessor(num_cols: list[str], cat_cols: list[str]) -> ColumnTransformer:
    transformers = []

    if num_cols:
        transformers.append(
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                num_cols,
            )
        )

    if cat_cols:
        transformers.append(
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                ]),
                cat_cols,
            )
        )

    return ColumnTransformer(transformers=transformers, remainder="drop")


def build_pipelines(preprocessor: ColumnTransformer, seed: int) -> dict[str, Pipeline]:
    return {
        "Logistic Regression": Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", LogisticRegression(max_iter=5000, solver="liblinear", random_state=seed)),
        ]),
        "KNN": Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", KNeighborsClassifier()),
        ]),
        "SVM": Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", SVC(probability=True, random_state=seed)),
        ]),
        "Random Forest": Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", RandomForestClassifier(random_state=seed)),
        ]),
        "Gradient Boosting": Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", GradientBoostingClassifier(random_state=seed)),
        ]),
    }


def grid_spaces() -> dict[str, dict]:
    return {
        "Logistic Regression": {
            "model__C": [0.01, 0.1, 1.0, 10.0, 50.0],
            "model__penalty": ["l2"],
            "model__solver": ["liblinear"],
        },
        "KNN": {
            "model__n_neighbors": [3, 5, 7, 9, 11, 15],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2],
        },
        "SVM": {
            "model__C": [0.1, 1.0, 10.0, 50.0],
            "model__kernel": ["linear", "rbf"],
            "model__gamma": ["scale", "auto", 0.01, 0.1],
        },
        "Random Forest": {
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [12, 24, 48, None],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
        },
        "Gradient Boosting": {
            "model__learning_rate": [0.05, 0.1, 0.15, 0.2],
            "model__n_estimators": [100, 200, 300],
            "model__max_depth": [2, 3, 4, 5],
            "model__subsample": [0.8, 1.0],
        },
    }


def random_spaces() -> dict[str, dict]:
    return {
        "Logistic Regression": {
            "model__C": loguniform(1e-3, 1e2),
            "model__penalty": ["l2"],
            "model__solver": ["liblinear"],
        },
        "KNN": {
            "model__n_neighbors": randint(3, 31),
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2],
        },
        "SVM": {
            "model__C": loguniform(1e-2, 1e2),
            "model__kernel": ["linear", "rbf"],
            "model__gamma": loguniform(1e-4, 1.0),
        },
        "Random Forest": {
            "model__n_estimators": randint(100, 500),
            "model__max_depth": randint(4, 60),
            "model__min_samples_split": randint(2, 20),
            "model__min_samples_leaf": randint(1, 10),
            "model__max_features": ["sqrt", "log2", None],
        },
        "Gradient Boosting": {
            "model__learning_rate": uniform(0.01, 0.29),
            "model__n_estimators": randint(50, 400),
            "model__max_depth": randint(2, 6),
            "model__subsample": uniform(0.6, 0.4),
        },
    }


def clean_params(best_params: dict) -> dict:
    out = {}
    for k, v in best_params.items():
        key = k.replace("model__", "")
        out[key] = v.item() if isinstance(v, np.generic) else v
    return out


def evaluate(model, X_test, y_test) -> dict:
    y_pred = model.predict(X_test)

    y_score = None
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        raw = model.decision_function(X_test)
        y_score = (raw - raw.min()) / (raw.max() - raw.min() + 1e-12)

    auc = np.nan
    if y_score is not None and len(np.unique(y_test)) > 1:
        auc = roc_auc_score(y_test, y_score)

    return {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "f1": float(f1_score(y_test, y_pred, zero_division=0)),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "kappa": float(cohen_kappa_score(y_test, y_pred)),
        "mcc": float(matthews_corrcoef(y_test, y_pred)),
    }


def run_search(method, pipelines, spaces, X_train, y_train, X_test, y_test, cv, seed, n_iter):
    rows = []
    best_models = {}
    best_params_by_model = {}

    for name, pipe in pipelines.items():
        if method == "grid":
            search = GridSearchCV(
                estimator=pipe,
                param_grid=spaces[name],
                scoring="accuracy",
                cv=cv,
                n_jobs=-1,
                refit=True,
            )
        else:
            search = RandomizedSearchCV(
                estimator=pipe,
                param_distributions=spaces[name],
                n_iter=n_iter,
                scoring="accuracy",
                cv=cv,
                random_state=seed,
                n_jobs=-1,
                refit=True,
            )

        search.fit(X_train, y_train)
        best = search.best_estimator_
        best_models[name] = best

        params = clean_params(search.best_params_)
        best_params_by_model[name] = params

        metrics = evaluate(best, X_test, y_test)
        row = {"model": name, "search_method": method, "best_params": json.dumps(params)}
        row.update(metrics)
        rows.append(row)

    return pd.DataFrame(rows), best_models, best_params_by_model


def run_stacking_grid(preprocessor, tuned_model_params, X_train, y_train, X_test, y_test, cv, seed):
    lr = LogisticRegression(max_iter=5000, solver="liblinear", random_state=seed)
    lr.set_params(**{k: v for k, v in tuned_model_params["Logistic Regression"].items() if k in lr.get_params()})

    knn = KNeighborsClassifier()
    knn.set_params(**{k: v for k, v in tuned_model_params["KNN"].items() if k in knn.get_params()})

    svm = SVC(probability=True, random_state=seed)
    svm.set_params(**{k: v for k, v in tuned_model_params["SVM"].items() if k in svm.get_params()})

    rf = RandomForestClassifier(random_state=seed)
    rf.set_params(**{k: v for k, v in tuned_model_params["Random Forest"].items() if k in rf.get_params()})

    gb = GradientBoostingClassifier(random_state=seed)
    gb.set_params(**{k: v for k, v in tuned_model_params["Gradient Boosting"].items() if k in gb.get_params()})

    stack = StackingClassifier(
        estimators=[("lr", lr), ("knn", knn), ("svm", svm), ("rf", rf), ("gb", gb)],
        final_estimator=LogisticRegression(max_iter=5000, solver="liblinear", random_state=seed),
        cv=cv,
        n_jobs=-1,
        stack_method="predict_proba",
    )

    pipe = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", stack),
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid={
            "model__final_estimator__C": [0.1, 1.0, 10.0],
            "model__final_estimator__penalty": ["l2"],
        },
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )

    grid.fit(X_train, y_train)
    best = grid.best_estimator_
    metrics = evaluate(best, X_test, y_test)

    best_params = {k.replace("model__", ""): v for k, v in grid.best_params_.items()}
    row = {"model": "Stacking", "search_method": "grid", "best_params": json.dumps(best_params)}
    row.update(metrics)

    return pd.DataFrame([row]), best


def save_diagnostics(best_models, X_test, y_test, fig_dir: Path, method: str):
    for model_name, model in best_models.items():
        y_pred = model.predict(X_test)

        fig, ax = plt.subplots(figsize=(5, 4))
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
        ax.set_title(f"Confusion Matrix - {model_name} ({method})")
        fig.tight_layout()
        fig.savefig(fig_dir / f"confusion_{method}_{model_name.replace(' ', '_').lower()}.png", dpi=200)
        plt.close(fig)

        if hasattr(model, "predict_proba") and len(np.unique(y_test)) > 1:
            y_score = model.predict_proba(X_test)[:, 1]
            fig2, ax2 = plt.subplots(figsize=(5, 4))
            RocCurveDisplay.from_predictions(y_test, y_score, ax=ax2)
            ax2.set_title(f"ROC - {model_name} ({method})")
            fig2.tight_layout()
            fig2.savefig(fig_dir / f"roc_{method}_{model_name.replace(' ', '_').lower()}.png", dpi=200)
            plt.close(fig2)


def save_metric_compare_plot(grid_df, random_df, fig_dir: Path):
    metrics = ["accuracy", "precision", "recall", "f1", "auc", "kappa", "mcc"]
    random_models = set(random_df["model"].tolist())

    for metric in metrics:
        g = grid_df[grid_df["model"].isin(random_models)][["model", metric]].copy()
        r = random_df[["model", metric]].copy()
        merged = g.merge(r, on="model", suffixes=("_grid", "_random")).sort_values("model")

        x = np.arange(len(merged))
        w = 0.35

        fig, ax = plt.subplots(figsize=(9, 5))
        ax.bar(x - w / 2, merged[f"{metric}_grid"], w, label="Grid Search")
        ax.bar(x + w / 2, merged[f"{metric}_random"], w, label="Random Search")
        ax.set_xticks(x)
        ax.set_xticklabels(merged["model"], rotation=20)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel(metric.upper())
        ax.set_title(f"Perbandingan {metric.upper()} (Grid vs Random)")
        ax.legend()
        fig.tight_layout()
        fig.savefig(fig_dir / f"metric_compare_{metric}.png", dpi=200)
        plt.close(fig)


def package_submission(root: Path, cfg: dict) -> Path:
    bundle = root / cfg["paths"]["submission_dir"]
    if bundle.exists():
        shutil.rmtree(bundle)
    bundle.mkdir(parents=True, exist_ok=True)

    include = [
        "data/raw/statlog_cleveland_hungary_final.csv",
        "results/tables/hasil_grid_search.csv",
        "results/tables/hasil_random_search.csv",
        "results/tables/data_profile.csv",
        "results/tables/run_summary.json",
    ]

    for rel in include:
        src = root / rel
        if src.exists():
            dst = bundle / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)

    fig_src = root / cfg["paths"]["figures"]
    if fig_src.exists():
        dst_fig = bundle / cfg["paths"]["figures"]
        shutil.copytree(fig_src, dst_fig, dirs_exist_ok=True)

    # try include notebook if found
    nb_candidates = list(Path('/content').glob('**/replikasi_metodologi.ipynb'))
    if nb_candidates:
        nb_src = sorted(nb_candidates, key=lambda p: len(str(p)))[0]
        shutil.copy2(nb_src, bundle / 'replikasi_metodologi.ipynb')

    zip_base = root / cfg["paths"]["submission_zip"].replace(".zip", "")
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=bundle))
    return zip_path


## 4) Run replication

In [8]:
cfg = default_cfg()
ensure_dirs(ROOT, cfg)

dataset_path, source_kind, source_ref = locate_or_download_dataset(ROOT, cfg)
print("Dataset:", dataset_path)
print("Source:", source_kind, source_ref)

df = pd.read_csv(dataset_path)
target_col = resolve_target_col(df, cfg["features"]["target_candidates"])
df[target_col] = coerce_binary_target(df[target_col])

profile = {
    "n_rows": int(df.shape[0]),
    "n_cols": int(df.shape[1]),
    "target_col": target_col,
    "missing_total": int(df.isna().sum().sum()),
    "class_distribution": df[target_col].value_counts(dropna=False).to_dict(),
}
pd.DataFrame([profile]).to_csv(ROOT / cfg["paths"]["tables"] / "data_profile.csv", index=False)

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

num_cols, cat_cols = resolve_feature_groups(X, cfg["features"]["categorical_candidates"])
print("Numeric features:", num_cols)
print("Categorical features:", cat_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=cfg["data"]["test_size"],
    random_state=cfg["project"]["random_state"],
    stratify=y,
)

preprocessor = build_preprocessor(num_cols, cat_cols)
pipelines = build_pipelines(preprocessor, cfg["project"]["random_state"])

cv = StratifiedKFold(
    n_splits=cfg["cv"]["folds"],
    shuffle=bool(cfg["cv"]["shuffle"]),
    random_state=cfg["project"]["random_state"],
)

grid_df, grid_models, tuned_params = run_search(
    "grid",
    pipelines,
    grid_spaces(),
    X_train,
    y_train,
    X_test,
    y_test,
    cv,
    cfg["project"]["random_state"],
    cfg["search"]["random_n_iter"],
)

if cfg["models"].get("enable_stacking_grid", True):
    stacking_df, stacking_model = run_stacking_grid(
        preprocessor,
        tuned_params,
        X_train,
        y_train,
        X_test,
        y_test,
        cv,
        cfg["project"]["random_state"],
    )
    grid_df = pd.concat([grid_df, stacking_df], ignore_index=True)
    grid_models["Stacking"] = stacking_model

random_df, random_models, _ = run_search(
    "random",
    pipelines,
    random_spaces(),
    X_train,
    y_train,
    X_test,
    y_test,
    cv,
    cfg["project"]["random_state"],
    cfg["search"]["random_n_iter"],
)

grid_csv = ROOT / cfg["paths"]["tables"] / "hasil_grid_search.csv"
random_csv = ROOT / cfg["paths"]["tables"] / "hasil_random_search.csv"

grid_df.to_csv(grid_csv, index=False)
random_df.to_csv(random_csv, index=False)

fig_dir = ROOT / cfg["paths"]["figures"]
save_diagnostics(grid_models, X_test, y_test, fig_dir, "grid")
save_diagnostics(random_models, X_test, y_test, fig_dir, "random")
save_metric_compare_plot(grid_df, random_df, fig_dir)

summary = {
    "dataset_path": str(dataset_path.relative_to(ROOT)),
    "dataset_sha256": sha256_file(dataset_path),
    "dataset_source_kind": source_kind,
    "dataset_source_ref": source_ref,
    "target_col": target_col,
    "n_rows": int(df.shape[0]),
    "n_cols": int(df.shape[1]),
    "numeric_features": num_cols,
    "categorical_features": cat_cols,
    "outputs": {
        "grid_csv": str(grid_csv.relative_to(ROOT)),
        "random_csv": str(random_csv.relative_to(ROOT)),
        "figures_dir": str(fig_dir.relative_to(ROOT)),
    },
}
summary_path = ROOT / cfg["paths"]["tables"] / "run_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

zip_path = package_submission(ROOT, cfg)
print("Done. Submission ZIP:", zip_path)
summary


Dataset: /content/task2_replicate/data/raw/statlog_cleveland_hungary_final.csv
Source: download https://raw.githubusercontent.com/mrdbourke/zero-to-mastery-ml/master/data/heart-disease.csv
Numeric features: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Categorical features: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal', 'ca']
Done. Submission ZIP: /content/task2_replicate/submission_bundle.zip


{'dataset_path': 'data/raw/statlog_cleveland_hungary_final.csv',
 'dataset_sha256': '7c3014365675306819510a49ff289efbec1d1a6a666a2dc7652f1547b383d859',
 'dataset_source_kind': 'download',
 'dataset_source_ref': 'https://raw.githubusercontent.com/mrdbourke/zero-to-mastery-ml/master/data/heart-disease.csv',
 'target_col': 'target',
 'n_rows': 303,
 'n_cols': 14,
 'numeric_features': ['age', 'trestbps', 'chol', 'thalach', 'oldpeak'],
 'categorical_features': ['sex',
  'cp',
  'fbs',
  'restecg',
  'exang',
  'slope',
  'thal',
  'ca'],
 'outputs': {'grid_csv': 'results/tables/hasil_grid_search.csv',
  'random_csv': 'results/tables/hasil_random_search.csv',
  'figures_dir': 'results/figures'}}

## 5) Preview main result tables

In [9]:
import pandas as pd
display(pd.read_csv(ROOT / 'results' / 'tables' / 'hasil_grid_search.csv'))
display(pd.read_csv(ROOT / 'results' / 'tables' / 'hasil_random_search.csv'))


,model,search_method,best_params,accuracy,precision,recall,f1,auc,kappa,mcc
0,Logistic Regression,grid,"{""C"": 0.1, ""penalty"": ""l2"", ""solver"": ""libline...",0.754098,0.736842,0.848485,0.788732,0.862554,0.498080,0.505201
1,KNN,grid,"{""n_neighbors"": 5, ""p"": 1, ""weights"": ""uniform""}",0.819672,0.761905,0.969697,0.853333,0.887446,0.627842,0.659142
2,SVM,grid,"{""C"": 1.0, ""gamma"": 0.1, ""kernel"": ""rbf""}",0.803279,0.756098,0.939394,0.837838,0.865801,0.595133,0.618072
3,Random Forest,grid,"{""max_depth"": 12, ""min_samples_leaf"": 2, ""min_...",0.819672,0.761905,0.969697,0.853333,0.903680,0.627842,0.659142
4,Gradient Boosting,grid,"{""learning_rate"": 0.05, ""max_depth"": 2, ""n_est...",0.803279,0.769231,0.909091,0.833333,0.899351,0.597360,0.609846
5,Stacking,grid,"{""final_estimator__C"": 1.0, ""final_estimator__...",0.803279,0.769231,0.909091,0.833333,0.886364,0.597360,0.609846


,model,search_method,best_params,accuracy,precision,recall,f1,auc,kappa,mcc
0,Logistic Regression,random,"{""C"": 0.033205591037519584, ""penalty"": ""l2"", ""...",0.737705,0.717949,0.848485,0.777778,0.853896,0.463146,0.472827
1,KNN,random,"{""n_neighbors"": 30, ""p"": 1, ""weights"": ""uniform""}",0.803279,0.744186,0.969697,0.842105,0.897186,0.592881,0.630261
2,SVM,random,"{""C"": 0.045821336908433555, ""gamma"": 0.0001154...",0.803279,0.756098,0.939394,0.837838,0.866883,0.595133,0.618072
3,Random Forest,random,"{""max_depth"": 42, ""max_features"": ""sqrt"", ""min...",0.819672,0.761905,0.969697,0.853333,0.900433,0.627842,0.659142
4,Gradient Boosting,random,"{""learning_rate"": 0.026844247528777843, ""max_d...",0.819672,0.775000,0.939394,0.849315,0.883117,0.629895,0.648128


## 6) Download submission ZIP

In [10]:
zip_path = ROOT / 'submission_bundle.zip'
print('ZIP path:', zip_path, 'exists=', zip_path.exists())

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Download otomatis tersedia saat dijalankan di Colab:', e)


ZIP path: /content/task2_replicate/submission_bundle.zip exists= True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>